In [8]:
import json
from typing import Iterable

import pandas as pd
import requests

In [16]:
def _fetch_json(url: str) -> list[list[object]]:
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    data = r.json()
    if not isinstance(data, list) or not data:
        raise ValueError(f"Unexpected JSON payload from {url}")
    return data

def _rows_to_frame(rows: list[list[object]]) -> pd.DataFrame:
    header = rows[0]
    values = rows[1:]
    df = pd.DataFrame(values, columns=header)
    for candidate in ["time_tag", "time_tag_utc", "time_tag_r"]:
        if candidate in df.columns:
            df.insert(0, column="timestamp", value=pd.to_datetime(df[candidate], utc=True, errors="coerce"))
            df = df.drop(candidate, axis=1)
            break
    if "timestamp" not in df.columns:
        raise ValueError(f"No time column in JSON header: {header}")
    df.set_index("timestamp")
    return df

plasma_response = _rows_to_frame(_fetch_json("https://services.swpc.noaa.gov/products/solar-wind/plasma-2-hour.json"))
print(plasma_response.head())

mag_response = _rows_to_frame(_fetch_json("https://services.swpc.noaa.gov/products/solar-wind/mag-2-hour.json"))
print(mag_response.head())


                  timestamp density  speed temperature
0 2026-04-20 06:55:00+00:00    3.27  441.8      112965
1 2026-04-20 06:56:00+00:00    4.76  438.6      100160
2 2026-04-20 06:57:00+00:00    3.82  427.1       73772
3 2026-04-20 06:58:00+00:00    3.80  423.3       68557
4 2026-04-20 06:59:00+00:00    4.83  435.6      117664
                  timestamp bx_gsm by_gsm bz_gsm lon_gsm lat_gsm    bt
0 2026-04-20 06:55:00+00:00   5.52  -3.56  -6.13  327.21  -43.01  8.98
1 2026-04-20 06:56:00+00:00   5.70  -3.19  -6.19  330.73  -43.45  9.00
2 2026-04-20 06:57:00+00:00   5.81  -2.90  -6.22  333.49  -43.79  9.00
3 2026-04-20 06:58:00+00:00   6.02  -2.50  -6.31  337.44  -44.07  9.08
4 2026-04-20 06:59:00+00:00   6.17  -1.71  -6.58  344.50  -45.75  9.18


In [17]:
def _coerce_numeric(df: pd.DataFrame, cols: Iterable[str]) -> pd.DataFrame:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

plasma = _coerce_numeric(plasma_response, ["density", "speed", "temperature"])
mag = _coerce_numeric(mag_response, ["bx_gsm", "by_gsm", "bz_gsm"])

print(plasma.head())
print(mag.head())

                  timestamp  density  speed  temperature
0 2026-04-20 06:55:00+00:00     3.27  441.8       112965
1 2026-04-20 06:56:00+00:00     4.76  438.6       100160
2 2026-04-20 06:57:00+00:00     3.82  427.1        73772
3 2026-04-20 06:58:00+00:00     3.80  423.3        68557
4 2026-04-20 06:59:00+00:00     4.83  435.6       117664
                  timestamp  bx_gsm  by_gsm  bz_gsm lon_gsm lat_gsm    bt
0 2026-04-20 06:55:00+00:00    5.52   -3.56   -6.13  327.21  -43.01  8.98
1 2026-04-20 06:56:00+00:00    5.70   -3.19   -6.19  330.73  -43.45  9.00
2 2026-04-20 06:57:00+00:00    5.81   -2.90   -6.22  333.49  -43.79  9.00
3 2026-04-20 06:58:00+00:00    6.02   -2.50   -6.31  337.44  -44.07  9.08
4 2026-04-20 06:59:00+00:00    6.17   -1.71   -6.58  344.50  -45.75  9.18


In [18]:
keep_plasma = [c for c in ["timestamp", "density", "speed", "temperature"] if c in plasma.columns]
keep_mag = [c for c in ["timestamp", "bx_gsm", "by_gsm", "bz_gsm"] if c in mag.columns]

merged = plasma[keep_plasma].merge(mag[keep_mag], on="timestamp", how="outer")
merged = merged.sort_values("timestamp").drop_duplicates(subset=["timestamp"]).reset_index(drop=True)

print(merged.head())

                  timestamp  density  speed  temperature  bx_gsm  by_gsm  \
0 2026-04-20 06:55:00+00:00     3.27  441.8       112965    5.52   -3.56   
1 2026-04-20 06:56:00+00:00     4.76  438.6       100160    5.70   -3.19   
2 2026-04-20 06:57:00+00:00     3.82  427.1        73772    5.81   -2.90   
3 2026-04-20 06:58:00+00:00     3.80  423.3        68557    6.02   -2.50   
4 2026-04-20 06:59:00+00:00     4.83  435.6       117664    6.17   -1.71   

   bz_gsm  
0   -6.13  
1   -6.19  
2   -6.22  
3   -6.31  
4   -6.58  


In [19]:
rename_map = {
    "bx_gsm": "BX_GSM",
    "by_gsm": "BY_GSM",
    "bz_gsm": "BZ_GSM",
    "speed": "V",
    "density": "N",
    "temperature": "T"
}
merged = merged.rename(columns=rename_map)

cols = [c for c in ["timestamp", "BX_GSM", "BY_GSM", "BZ_GSM", "V", "N", "T"] if c in merged.columns]

dscovr_observations = merged[cols]

print(dscovr_observations.head())

                  timestamp  BX_GSM  BY_GSM  BZ_GSM      V     N       T
0 2026-04-20 06:55:00+00:00    5.52   -3.56   -6.13  441.8  3.27  112965
1 2026-04-20 06:56:00+00:00    5.70   -3.19   -6.19  438.6  4.76  100160
2 2026-04-20 06:57:00+00:00    5.81   -2.90   -6.22  427.1  3.82   73772
3 2026-04-20 06:58:00+00:00    6.02   -2.50   -6.31  423.3  3.80   68557
4 2026-04-20 06:59:00+00:00    6.17   -1.71   -6.58  435.6  4.83  117664
